In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/dlp-image-classification-apr-2026-2/sample_submission.csv
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_06930.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_07759.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_02911.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_05770.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_00433.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_07838.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_02292.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_06343.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_01971.jpg
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_08741.JPG
/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test/image_0

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
TRAIN_DIR = "/kaggle/input/competitions/dlp-image-classification-apr-2026-2/train"
TEST_DIR = "/kaggle/input/competitions/dlp-image-classification-apr-2026-2/test"

In [4]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [5]:
full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)

# Save class mapping
class_to_idx = full_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("Number of classes:", len(class_to_idx))

# Train/Val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply validation transforms separately
val_dataset.dataset.transform = val_transforms

Number of classes: 38


In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [7]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 38)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 131MB/s] 


In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

In [9]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    
    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [10]:
def validate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
    
    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [11]:
best_acc = 0

for epoch in range(10):
    print(f"\nEpoch {epoch+1}")
    
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)
    
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")


Epoch 1


100%|██████████| 1086/1086 [03:12<00:00,  5.65it/s]


Train Loss: 1.0633, Acc: 0.9024
Val   Loss: 0.7262, Acc: 0.9919

Epoch 2


100%|██████████| 1086/1086 [03:13<00:00,  5.63it/s]


Train Loss: 0.7365, Acc: 0.9897
Val   Loss: 0.7011, Acc: 0.9945

Epoch 3


...

Train Loss: 0.6947, Acc: 0.9981
Val   Loss: 0.6883, Acc: 0.9971

Epoch 9


100%|██████████| 1086/1086 [03:11<00:00,  5.66it/s]


Train Loss: 0.6947, Acc: 0.9977
Val   Loss: 0.6936, Acc: 0.9956

Epoch 10


100%|██████████| 1086/1086 [03:11<00:00,  5.66it/s]


Train Loss: 0.6943, Acc: 0.9977
Val   Loss: 0.6861, Acc: 0.9977


In [12]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [13]:
test_transforms = val_transforms

image_ids = []
predictions = []

for img_name in tqdm(os.listdir(TEST_DIR)):
    img_path = os.path.join(TEST_DIR, img_name)
    
    image = Image.open(img_path).convert("RGB")
    image = test_transforms(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(image)
        pred = output.argmax(1).item()
    
    image_ids.append(img_name.replace(".jpg", ""))
    predictions.append(pred)

100%|██████████| 10876/10876 [03:04<00:00, 59.02it/s]


In [14]:
df = pd.DataFrame({
    "Image_ID": image_ids,
    "Label": predictions
})

df["Image_ID"] = df["Image_ID"].str.replace(".JPG", "", regex=False)
df["Image_ID"] = df["Image_ID"].str.replace(".jpg", "", regex=False)
df = df.sort_values("Image_ID")

df.to_csv("submission.csv", index=False)
print("Submission file saved!")

Submission file saved!
